<a href="https://colab.research.google.com/github/Valeriaya512/R/blob/main/%E9%97%9C%E8%81%AF%E8%B3%87%E6%96%99%E5%BA%AB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Primary Key
用來唯一識別資料表中每一筆資料的欄位，通常不能重複，也不能是空值
*  每一筆資料都要能被唯一找到
*  不能重複，也不能是NULL
*  一張表通常會有一個主鍵，可能是單一欄位，也可能是複合欄位

## Foreign Key
用來連結兩個資料表的欄位，它的值通常會對應到另一張表的主鍵，目的是維持資料一致性與參照完整性
*  用來表示表與表之間的關係
*  要能對應到另一張表的主鍵，否則可能會違反資料完整性
*  可以幫助避免插入不存在的參照資料


In [3]:
library(tidyverse)

students <- tibble(
  student_id = c(1, 2, 3),   #Primary Key
  student_name = c("Amy", "Ben", "Cathy")
)

orders <- tibble(
  order_id = c(101, 102, 103, 104),
  student_id = c(1, 1, 2, 3),  #Foreign Key
  course = c("R", "SQL", "Python", "Statistics")
)

students
orders

student_id,student_name
<dbl>,<chr>
1,Amy
2,Ben
3,Cathy


order_id,student_id,course
<dbl>,<dbl>,<chr>
101,1,R
102,1,SQL
103,2,Python
104,3,Statistics


## join：連接表格

Tidy架構中連結兩張表的方式主要有以下兩種：
*  mutating jion：將兩張表結合，並在新的表格中納入兩張表格有的變數
*  filtering jion：透過另一張表的資訊篩選目標資料表格的rows

常見的join有：
*  inner_join()：只保留兩張表都能配對到的資料，像交集
*  left_join()：保留左表全部資料，右表配對到的欄位補進來；配不到就會是 NA
*  right_join()：保留右表全部資料，左表配對到的欄位補進來；配不到就會是 NA
*  full_join()：兩張表所有資料都保留，配不到的地方補 NA
*  semi_join()：只篩選左表中「有在右表出現過」的列，不會把右表欄位接進來
*  anti_join()：只保留左表中「沒有在右表出現過」的列
*  cross_join()：兩張表做笛卡兒積，常用在需要所有可能組合的情況

常見應用：
*  查主檔加明細：例如學生主檔搭配選課資料，通常用 left_join()
*  找共同資料：例如只分析兩張表都存在的顧客，適合 inner_join()
*  找缺漏資料：例如找出訂單表中有、但主檔沒有的編號，可用 anti_join()
*  先篩選再分析：例如只留下在某個名單中的資料，可用 semi_join()

In [15]:
#範例
library(tidyverse)

students <- tibble(
  student_id = c(1, 2, 3),
  name = c("Amy", "Ben", "Cathy")
)

orders <- tibble(
  student_id = c(1, 1, 2, 4),
  course = c("R", "SQL", "Python", "Excel")
)

students
orders

orders1 <- orders %>%
  inner_join(students, by = "student_id")
orders2 <- orders %>%
  left_join(students, by = "student_id")
orders3 <- orders %>%
  right_join(students, by = "student_id")
orders4 <- orders %>%
  full_join(students, by = "student_id")
orders5 <- orders %>%
  semi_join(students, by = "student_id")
orders6 <- orders %>%
  anti_join(students, by = "student_id")

print(orders1)
print(orders2)
print(orders3)
print(orders4)
print(orders5)
print(orders6)

student_id,name
<dbl>,<chr>
1,Amy
2,Ben
3,Cathy


student_id,course
<dbl>,<chr>
1,R
1,SQL
2,Python
4,Excel


# A tibble: 3 × 3
  student_id course name 
       <dbl> <chr>  <chr>
1          1 R      Amy  
2          1 SQL    Amy  
3          2 Python Ben  
# A tibble: 4 × 3
  student_id course name 
       <dbl> <chr>  <chr>
1          1 R      Amy  
2          1 SQL    Amy  
3          2 Python Ben  
4          4 Excel  NA   
# A tibble: 4 × 3
  student_id course name 
       <dbl> <chr>  <chr>
1          1 R      Amy  
2          1 SQL    Amy  
3          2 Python Ben  
4          3 NA     Cathy
# A tibble: 5 × 3
  student_id course name 
       <dbl> <chr>  <chr>
1          1 R      Amy  
2          1 SQL    Amy  
3          2 Python Ben  
4          4 Excel  NA   
5          3 NA     Cathy
# A tibble: 3 × 2
  student_id course
       <dbl> <chr> 
1          1 R     
2          1 SQL   
3          2 Python
# A tibble: 1 × 2
  student_id course
       <dbl> <chr> 
1          4 Excel 


In [21]:
# 欄位名稱不一樣
library(tidyverse)

# 學生表：欄位是 student_id
students <- tibble(
  student_id = c(1, 2, 3),
  name = c("Amy", "Ben", "Cathy")
)

# 訂單表：欄位是 id（與學生表的 student_id 不同）
orders <- tibble(
  id = c(1, 1, 2, 4),
  course = c("R", "SQL", "Python", "Excel")
)

orders1 <- orders %>%
  inner_join(students, by = c("id" = "student_id"))
orders2 <- orders %>%
  left_join(students, by = c("id" = "student_id"))

print(orders1)
print(orders2)

# A tibble: 3 × 3
     id course name 
  <dbl> <chr>  <chr>
1     1 R      Amy  
2     1 SQL    Amy  
3     2 Python Ben  
# A tibble: 4 × 3
     id course name 
  <dbl> <chr>  <chr>
1     1 R      Amy  
2     1 SQL    Amy  
3     2 Python Ben  
4     4 Excel  NA   


In [22]:
# 一對多或多對一

library(tidyverse)

students <- tibble(
  student_id = c(1, 2, 3),
  name = c("Amy", "Ben", "Cathy")
)

orders <- tibble(
  student_id = c(1, 1, 2, 4),
  course = c("R", "SQL", "Python", "Excel")
)

# student_id = 1 在 orders 中有兩筆（R 和 SQL）
left_join(students, orders, by = "student_id")

student_id,name,course
<dbl>,<chr>,<chr>
1,Amy,R
1,Amy,SQL
2,Ben,Python
3,Cathy,NA


In [24]:
# 檢查key值是否唯一，">1"表示不唯一
students %>% count(student_id)
orders %>% count(student_id)

# 用 distincy() 去重
orders_unique <- orders %>%
  distinct(student_id, .keep_all = TRUE)

left_join(students, orders_unique, by = "student_id")

# 使用 group_by() + summarize() 彙總
orders_first <- orders %>%
  arrange(student_id, course) %>%
  group_by(student_id) %>%
  slice_head(n = 1) %>%
  ungroup()

left_join(students, orders_first, by = "student_id")

# 使用 semi_join() / anti_join() 避免行數膨脹
# semi_join()：只篩選，不會帶入右表欄位，不會增加行數
# anti_join()：只保留沒有配對的行數，也不會增加行數

student_id,n
<dbl>,<int>
1,1
2,1
3,1


student_id,n
<dbl>,<int>
1,2
2,1
4,1


student_id,name,course
<dbl>,<chr>,<chr>
1,Amy,R
2,Ben,Python
3,Cathy,NA


student_id,name,course
<dbl>,<chr>,<chr>
1,Amy,R
2,Ben,Python
3,Cathy,NA


## 集合運算
*  intersect(x,y)：回傳同時出現在x與y的觀察個體
*  union(x,y)：回傳有出現在x或y的觀察個體，且個體/row不會重複
*  setdiff(x,y)：回傳有出現在x但沒有出現在y的觀察個體